In [2]:
!pip install -q transformers datasets accelerate

import random
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight
from google.colab import files

#sets the seed for reproducibility
SEED = 777
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    from transformers import set_seed
    set_seed(SEED)
except ImportError:
    pass

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

MODEL_NAME = "microsoft/deberta-v3-large"
OUTPUT_DIR = "./deberta_v3_final_submission"
MAX_LEN = 512
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
LR = 8e-6
EPOCHS = 12

print("Loading Data...")
dataset = load_dataset("ailsntua/QEvasion")

def preprocess_text(example):
    # input handling
    clarity = example.get('clarity_label', 'Unknown')
    if clarity is None:
        clarity = "Unknown"

    text = f"Context: {clarity} | Question: {example['question']} Answer: {example['interview_answer']}"

    # handle labels safely for Test Set
    label = example.get("evasion_label", -1)

    return {"text": text, "evasion_label": label}

print("Preprocessing Training Data...")
train_ds = dataset["train"].map(preprocess_text)

print("Preprocessing Test Data...")
if "test" in dataset:
    test_ds = dataset["test"].map(preprocess_text)
else:
    raise ValueError("Test set not found!")

# encode labels
train_ds = train_ds.class_encode_column("evasion_label")
labels = train_ds.features["evasion_label"].names
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

print("Tokenizing...")
train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

# set labels for Trainer
train_ds = train_ds.map(lambda x: {"labels": x["evasion_label"]})
# impliments a weighted trainer
y_train = train_ds["evasion_label"]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float
).to("cuda" if torch.cuda.is_available() else "cpu")

class ProTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)

        if hasattr(outputs, "logits"):
            logits = outputs.logits
        else:
            logits = outputs[1]
        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights_tensor,
            label_smoothing=0.1
        )

        loss = loss_fct(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1)
        )
        return (loss, outputs) if return_outputs else loss

# training
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    num_train_epochs=EPOCHS,

    weight_decay=0.05,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    # CRITICAL: No evaluation strategy because we have no dev set
    eval_strategy="no",
    save_strategy="no",
    load_best_model_at_end=False,

    fp16=True,
    report_to="none",
    dataloader_num_workers=2,
    seed=SEED
)

trainer = ProTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    # uses 100% of data for training
)

print(f"Starting FULL TRAINING on {len(train_ds)} examples for {EPOCHS} Epochs...")
trainer.train()
print("\nGenerating final submission.csv...")

if "index" not in test_ds.column_names:
    test_ds = test_ds.add_column("index", range(len(test_ds)))

test_preds = trainer.predict(test_ds)
pred_ids = np.argmax(test_preds.predictions, axis=-1)
pred_labels = [id2label[p] for p in pred_ids]

out_df = pd.DataFrame({
    "index": test_ds["index"],
    "evasion_label": pred_labels
})
out_df.to_csv("submission_pro.csv", index=False)
print("Done! Downloading...")
files.download("submission_pro.csv")

Loading Data...
Preprocessing Training Data...


Map:   0%|          | 0/3448 [00:00<?, ? examples/s]

Preprocessing Test Data...


Map:   0%|          | 0/308 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/3448 [00:00<?, ? examples/s]

...

Step,Training Loss
500,2.095200
1000,1.729400
1500,1.497800
2000,1.265800
2500,1.174300



Generating final submission.csv...


Done! Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
df = pd.read_csv("submission_pro.csv")

# count how many times each label appears
print(df["evasion_label"].value_counts())

evasion_label
Explicit               79
General                63
Dodging                50
Implicit               45
Deflection             42
Declining to answer    13
Partial/half-answer     6
Claims ignorance        6
Clarification           4
Name: count, dtype: int64
